In [ ]:
from ast import literal_eval

import pandas as pd


movies = pd.read_csv("D:/Areeb/Projects 101/Movie_Recommender/data/raw/tmdb_5000_movies.csv")
credits = pd.read_csv("D:/Areeb/Projects 101/Movie_Recommender/data/raw/tmdb_5000_credits.csv")

movies.head()


print(movies.shape)
print(credits.shape)

credits.head()

movies.head()
movies.columns


movies.loc[0]

credits.head()

movies_merged =  movies.merge(credits, right_on="movie_id", left_on="id")
print(movies_merged.shape)

print(movies_merged.columns)

movies_merged.head()
#cleaning left droping the cols  
updated_merged = movies_merged.drop(columns=['budget', 'homepage',
       'original_title',  'popularity', 'production_companies',
       'production_countries','original_language', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average',
       'vote_count', 'movie_id', 'title_y'])



updated_merged = updated_merged.rename(columns={'title_x': 'title'})
print(updated_merged.columns)
updated_merged.iloc[0]


#literal_eval converts string representation of list/dict to actual list/dict

def convert(obj):
    L = []
    
    for i in literal_eval(obj):
        L.append(i['name'].replace(" ", ""))
    return L  

def convert_cast(obj):
    L = []
   
    for i in literal_eval(obj):
        if len(L) < 3:
            L.append(i['name'].replace(" ", ""))
        else:
            break    
    return L  

def convert_crew(obj):
    L = []
    
    for i in literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'].replace(" ", ""))
    return L  

updated_merged['genres'] = updated_merged['genres'].apply(convert)
print(updated_merged['genres'].head())

updated_merged['cast'] = updated_merged['cast'].apply(convert_cast)
print(updated_merged['cast'].head())

updated_merged['keywords'] = updated_merged['keywords'].apply(convert)
print(updated_merged['keywords'].head())

updated_merged['crew'] = updated_merged['crew'].apply(convert_crew)
print(updated_merged['crew'].head())

updated_merged['overview'] = updated_merged['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])
print(updated_merged['overview'].head())

updated_merged['tags'] = updated_merged['genres'] + updated_merged['keywords'] + updated_merged['cast'] + updated_merged['crew'] + updated_merged['overview']
print(updated_merged['tags'].head())

updated_merged['tags'] = updated_merged['tags'].apply(lambda x: " ".join(x).lower())
updated_merged.head()









(4803, 20)
(4803, 4)
(4803, 24)
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='str')
Index(['genres', 'id', 'keywords', 'overview', 'title', 'cast', 'crew'], dtype='str')
0    [Action, Adventure, Fantasy, ScienceFiction]
1                    [Adventure, Fantasy, Action]
2                      [Action, Adventure, Crime]
3                [Action, Crime, Drama, Thriller]
4             [Action, Adventure, ScienceFiction]
Name: genres, dtype: object
0    [SamWorthington, ZoeSaldana, SigourneyWeaver]
1       [JohnnyDepp, OrlandoBloom, KeiraKnightley]
2        [DanielCraig, ChristophWaltz, LéaSeydoux]
3        [ChristianBale, MichaelCaine, GaryOldman]
4      [TaylorKitsch, L

,genres,id,keywords,overview,title,cast,crew,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],action adventure fantasy sciencefiction cultur...
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",Pirates of the Caribbean: At World's End,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],adventure fantasy action ocean drugabuse exoti...
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...","[A, cryptic, message, from, Bond’s, past, send...",Spectre,"[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],action adventure crime spy basedonnovel secret...
3,"[Action, Crime, Drama, Thriller]",49026,"[dccomics, crimefighter, terrorist, secretiden...","[Following, the, death, of, District, Attorney...",The Dark Knight Rises,"[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],action crime drama thriller dccomics crimefigh...
4,"[Action, Adventure, ScienceFiction]",49529,"[basedonnovel, mars, medallion, spacetravel, p...","[John, Carter, is, a, war-weary,, former, mili...",John Carter,"[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],action adventure sciencefiction basedonnovel m...


In [2]:
#Stemming
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ps = PorterStemmer()

def convert_tags(obj):
    L = []
    
    for word in obj.split():
        L.append(ps.stem(word))
    return " ".join(L)

updated_merged['tags_stemmed'] = updated_merged['tags'].apply(convert_tags)
#updated_merged['tags_stemmed'].iloc[0]

#Vocab
cv = CountVectorizer(
    max_features=5000, #keeping only 5000 most frequent words
    stop_words='english' #this remove common words like a, the, is, are etc
)
#fit learns the vocab, transform converts every movie to vector and toarray converts it to array
vectors = cv.fit_transform(updated_merged['tags_stemmed']).toarray()
#print(vectors.shape)
#type(vectors)


#Cosine Similarity
similarity = cosine_similarity(vectors)
#print(similarity.shape)
#print(max(similarity[0]))

def recommend(movie):
    index = updated_merged[updated_merged['title'] == movie]
    if index.empty:
        return "Movie not found in the dataset."
    else:
        index = index.index[0]
        
    similar_movies = similarity[index]
    movie_list = list(enumerate(similar_movies))
    movie_list = sorted(movie_list, key=lambda x: x[1], reverse=True)[1:6]
    return [(updated_merged.iloc[idx].title, 
             round(float(score)*100,1)) 
             for idx,score in movie_list]

print(recommend("Avatar"))




[('Aliens vs Predator: Requiem', 28.7), ('Aliens', 26.9), ('Falcon Rising', 26.1), ('Independence Day', 25.6), ('Titan A.E.', 25.0)]


In [14]:
import pickle
with open('D:/Areeb/Projects 101/Movie_Recommender/models/movies.pkl', 'wb') as file:
    pickle.dump(updated_merged, file)
with open('D:/Areeb/Projects 101/Movie_Recommender/models/similarity.pkl', 'wb') as file:
    pickle.dump(similarity, file)

pickle_movies = pickle.load(open('D:/Areeb/Projects 101/Movie_Recommender/models/movies.pkl', 'rb'))
pikcle_similarity = pickle.load(open('D:/Areeb/Projects 101/Movie_Recommender/models/similarity.pkl', 'rb')) 
print(pickle_movies.shape)
print(pikcle_similarity.shape)

print (pickle_movies.columns.to_list())

(4803, 9)
(4803, 4803)
['genres', 'id', 'keywords', 'overview', 'title', 'cast', 'crew', 'tags', 'tags_stemmed']
